# duckdb-text2sql: LoRA fine-tuning on Kaggle

Fine-tunes Qwen3.5-0.8B with LoRA on execution-verified DuckDB text-to-SQL data
([bhargavis/spider-duckdb](https://huggingface.co/datasets/bhargavis/spider-duckdb) plus the DuckDB-specific
examples generated by the repo), then evaluates the base and fine-tuned models by execution.

Settings before running: Accelerator **GPU T4 x2** (one GPU is used), Internet **on**.
To push the adapter to the Hub, add a Kaggle secret named `HF_TOKEN` (a Hugging Face write token).

All the logic lives in `src/text2sql/sft.py`; this notebook only sets up the environment and calls it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/bhargaviHQ/huggingface-projects.git
%cd huggingface-projects/duckdb-text2sql
!git log -1 --format='commit %h %s'
!pip install -q -r requirements-train.txt -e .

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as e:
    print("no HF_TOKEN secret; training works, but --push will fail:", type(e).__name__)

## 1. Smoke test (about 10 minutes)

10 optimizer steps and 8 questions per eval set. Checks that the model loads, the loss is finite and
evaluation runs. The accuracy numbers here mean nothing.

In [ ]:
!python -m text2sql.sft --smoke

## 2. Full run

2 epochs over about 7,700 examples, then evaluation of the base and fine-tuned models on 543 questions.
`--push` uploads the adapter and results to a private Hub repo. If the session might time out, use
**Save Version > Save & Run All** instead, which keeps running after the browser is closed.

In [ ]:
!python -m text2sql.sft --push

In [ ]:
import glob, json
latest = sorted(p for p in glob.glob("results/sft_*.json") if "smoke" not in p)[-1]
r = json.load(open(latest))
print(latest, r["gpu"], "train examples:", r["train_examples"])
for name in r["base"]:
    print(f'{name:<12} base {r["base"][name]["execution_accuracy"]:>5}%   fine-tuned {r["fine_tuned"][name]["execution_accuracy"]:>5}%')